In [ ]:
def ensure_columns(df: DataFrame, schema_map: dict[str, T.DataType]) -> DataFrame:
    """
    Add optional missing columns as nulls, and cast present columns to expected types.
    """
    out = df
    for c, dtype in schema_map.items():
        if c not in out.columns:
            out = out.withColumn(c, F.lit(None).cast(dtype))
        else:
            out = out.withColumn(c, F.col(c).cast(dtype))
    return out

def add_derived_columns(df: DataFrame) -> DataFrame:
    """
    Mirror the spirit of your Pandas PreEDA:
    - convert integer epoch columns to timestamps
    - add review_length
    - add a few quality flags
    - add log1p helper columns for skewed counts
    """
    out = df

    # Timestamp conversions
    out = out.withColumn("author_last_played_ts", F.to_timestamp(F.from_unixtime("author_last_played")))
    out = out.withColumn("timestamp_created_ts", F.to_timestamp(F.from_unixtime("timestamp_created")))
    out = out.withColumn("timestamp_updated_ts", F.to_timestamp(F.from_unixtime("timestamp_updated")))

    # Text length
    out = out.withColumn(
        "review_length",
        F.when(F.col("review").isNull(), None).otherwise(F.length("review"))
    )
    
    # Mild derived features aligned to your sample notebook ideas
    for src, dst in [
        ("votes_up", "log_votes_up"),
        ("votes_funny", "log_votes_funny"),
        ("comment_count", "log_comment_count"),
        ("author_num_reviews", "log_author_num_reviews"),
        ("author_num_games_owned", "log_author_num_games_owned"),
        ("author_playtime_forever", "log_author_playtime_forever"),
        ("author_playtime_last_two_weeks", "log_author_playtime_last_two_weeks"),
        ("author_playtime_at_review", "log_author_playtime_at_review"),
        ("review_length", "log_review_length"),
    ]:
        out = out.withColumn(
            dst,
            F.when(F.col(src).isNull(), None).otherwise(F.log1p(F.col(src)))
        )

    return out

